# Part 2: Unit conversions and `mutate`

Reminder: (reword this paragraph)
We take the average quadrat biomass at each site because due to the large size of the sites, it would be extremely time consuming and expensive to get an actual total biomass of each site. This average gives us an idea of the average biomass in each of the 50 quadrats where data was taken. This data is currently in g/m^2, but the site area data has been given in different units-- hectares. Bison also consume a lot of plants, with a single bison needing about 21,900 kg (1 kg = 1000g) of biomass a year to live a healthy life. So, we have data in different units from one another. Fortunately, we can easily do math in R to create new variables with updated units. 

Here are all of the unit conversions that need to happen: 
  
1. currently in g/0.25 m2 and want to get to kg/ha 
2. Conversion of g/0.25 m2 to kg/ha is to multiply by 40

To do this conversion and add it to our data, we need to make a new column with the converted units. To make a new column in a dataset, we can use the `mutate()` function in tidyverse. `mutate()` creates new columns that are functions of existing variables. It can also modify and delete columns, but we won't do either of those today. To make sure that this does not alter the original data, we will make a copy of the biomass dataset with the new column and assign it to a new dataset (using <-) called "biomass_kgha". Since this is just adding a new column to the original data, you can also apply these changes to "biomass" and not make a new dataset by doing "biomass <-" instead. Just make sure to be careful when making changes to any dataset!

## Converting units to kg/ha 

In [3]:
library(tidyverse)
biomass <- read.csv("biomass.csv", header=T, sep=',')

biomass_kgha <- mutate(biomass, kg_ha = Biomass * 40) 
#the various parts of the mutate() function: mutate(name of dataset, new column = function of existing column)

head(biomass_kgha) #preview the dataframe with the new column  


,Site,Quadrat,Biomass,kg_ha
,<int>,<int>,<dbl>,<dbl>
1,1,1,55.51619,2220.648
2,1,2,58.15858,2326.343
3,1,3,72.46967,2898.787
4,1,4,60.56407,2422.563
5,1,5,61.03430,2441.372
6,1,6,73.72052,2948.821


This new dataset `biomass_kgha` now has a new column that converts the biomass in $g/m^2$ into kg/ha. The column "kg_ha" will be the new column we will call when we are looking at the biomass data because its now in the correct units!

We should take the average biomass per quadrat again, but this time with the converted units.

### Find the new average biomass

Repeat `group_by()` and `summarize()` steps from part 1. 

In [4]:
biomass_avg <- biomass_kgha %>% 
  group_by(Site) %>% 
  summarize(avg_biomass=mean(kg_ha))

biomass_avg

Site,avg_biomass
<int>,<dbl>
1,2411.009
2,4117.127
3,5098.440


We have just used two different ways of manipulating and creating data in R: the `summarise()` and `mutate()` functions. 
They are very similar to each other, but have some key differences from one another. One of the main 
differences between them is that `summarise()` will only produce a single value per group,
while `mutate()` produces the same number of rows as the input. 

If you'd like, play around with both `mutate()` and `group_by() %>% summarise()` later on in this module to get a clearer idea of how they differ from one another

### Join datasets

There is another dataset the tribal agriculture managers have with the sizes of each site in hectares, called "sites", which we loaded in with biomass data at the beginning of class. We are going to merge this "sites" dataset with our `biomass_avg` dataset, which has the average biomass at each site (kg per hectare), into one dataset using the function left_join(), which we learned in the Water Module. We can do this because both datasets have matching columns called Site we can use as the key. 

In [5]:
sites <- read.csv("bison_sites.csv", header=T, sep=',')
site_biomass <- left_join(sites, biomass_avg, by = "Site") #left_join(dataset 1, dataset 2, by = "shared column"); Join matching rows from dataset 2 to dataset 1 using their shared column "Site".
site_biomass

Site,Hectares,avg_biomass
<int>,<int>,<dbl>
1,1700,2411.009
2,1300,4117.127
3,900,5098.440


🧠✍️**Class Questions**

* How does merging the datasets help us see the data?

* What can this new tibble tell us about the quantity of biomass in each site?

* Just from viewing this dataset, do you have a hypothesis of which site might be most suitable to reintroduce bison?


To get a rough estimate of the total available biomass in each site we should multiply the average biomass per quadrat (kg/ha) by the hectares of each site (ha). This will give us a total kg of biomass at each site. This should be added as a new column to the site_biomass tibble.

### Estimate total available biomass

In [9]:
total_available <- mutate(site_biomass, total_biomass = Hectares * avg_biomass)
total_available

Site,Hectares,avg_biomass,total_biomass
<int>,<int>,<dbl>,<dbl>
1,1700,2411.009,4098716
2,1300,4117.127,5352265
3,900,5098.440,4588596


🧠✍️**Class Question**

* What can these totals tell us about each site? Which site do you think is best suited for bison reintroduction? Why?

## Lesson Recap 